In [23]:
# 导入库
import os
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel, AdamW
use_gpu = torch.cuda.is_available()
print(use_gpu)
# 检测设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

False
Device: cpu


In [24]:
# 数据集定义
class Dataset(torch.utils.data.Dataset):
    def __init__(self, file_path):
        dataset = pd.read_excel(file_path)
        
        # 数据清洗步骤
        dataset = dataset.dropna(subset=['content'])  # 删除内容为空的行
        dataset = dataset[dataset['content'].apply(lambda x: isinstance(x, str))]  # 确保内容是字符串类型
        
        train, test = train_test_split(dataset, test_size=0.2, random_state=123)
        self.train = train.reset_index(drop=True)

    def __len__(self):
        return len(self.train)

    def __getitem__(self, index):
        label = self.train.loc[index, 'label']
        text = self.train.loc[index, 'content']
        return label+1, text

In [25]:
# 数据加载器中的处理函数
def collate_fn(data):
    sents = [item[1] for item in data]
    labels = [item[0] for item in data]

    data = tokenizer.batch_encode_plus(
        batch_text_or_text_pairs=sents,
        truncation=True,
        padding='max_length',
        max_length=500,
        return_tensors='pt',
    )

    input_ids = data['input_ids'].to(device)
    attention_mask = data['attention_mask'].to(device)
    token_type_ids = data['token_type_ids'].to(device)
    labels = torch.LongTensor(labels).to(device)

    return input_ids, attention_mask, token_type_ids, labels

In [26]:
# 从清华大学的镜像源加载模型和分词器
model_name = "bert-base-chinese"
tokenizer = BertTokenizer.from_pretrained(model_name, mirror='https://mirrors.tuna.tsinghua.edu.cn/hugging-face-models/')
pretrained_model = BertModel.from_pretrained(model_name, mirror='https://mirrors.tuna.tsinghua.edu.cn/hugging-face-models/').to(device)

print("BERT tokenizer and model loaded successfully!")

BERT tokenizer and model loaded successfully!


In [27]:
# 下游任务模型定义
class SentimentClassifier(torch.nn.Module):
    def __init__(self):
        super(SentimentClassifier, self).__init__()
        self.rnn = torch.nn.LSTM(768, 768, bidirectional=True, batch_first=True)
        self.fc = torch.nn.Linear(768 * 2, 3)

    def forward(self, input_ids, attention_mask, token_type_ids):
        with torch.no_grad():
            bert_output = pretrained_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        lstm_out, _ = self.rnn(bert_output.last_hidden_state[:, 0].unsqueeze(1))
        logits = self.fc(lstm_out.squeeze(1))
        return torch.nn.functional.softmax(logits, dim=1)

In [28]:
# 数据加载器
file_path = '/Users/hezixin/Downloads/新能源汽车政策评论打标数据.xlsx'  # 上传数据文件到 Colab
train_dataset = Dataset(file_path)
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=16,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True
)

In [29]:
# 实例化模型
model = SentimentClassifier().to(device)

In [30]:
# 训练设置
optimizer = AdamW(model.parameters(), lr=5e-5)# 使用PyTorch自带的AdamW优化器
criterion = torch.nn.CrossEntropyLoss()

/opt/anaconda3/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [31]:
# 模型训练
def train_model(num_epochs=5):
    model.train()
    for epoch in range(num_epochs):
        for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(train_loader):
            outputs = model(input_ids, attention_mask, token_type_ids)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if i % 5 == 0:
                preds = outputs.argmax(dim=1)
                accuracy = (preds == labels).float().mean().item()
                print(f'Epoch {epoch}, Step {i}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')

In [32]:
# 执行训练
train_model()

Epoch 0, Step 0, Loss: 1.0974, Accuracy: 0.5000
Epoch 0, Step 5, Loss: 1.0790, Accuracy: 0.4375
Epoch 0, Step 10, Loss: 1.0860, Accuracy: 0.3750
Epoch 0, Step 15, Loss: 1.0605, Accuracy: 0.4375
Epoch 0, Step 20, Loss: 1.0764, Accuracy: 0.3750
Epoch 0, Step 25, Loss: 1.0673, Accuracy: 0.3125
Epoch 0, Step 30, Loss: 1.0375, Accuracy: 0.2500
Epoch 0, Step 35, Loss: 1.0581, Accuracy: 0.3125
Epoch 0, Step 40, Loss: 1.0872, Accuracy: 0.3125
Epoch 0, Step 45, Loss: 1.0466, Accuracy: 0.5625
Epoch 0, Step 50, Loss: 1.0269, Accuracy: 0.3125
Epoch 0, Step 55, Loss: 1.0542, Accuracy: 0.4375
Epoch 0, Step 60, Loss: 1.0256, Accuracy: 0.5000
Epoch 0, Step 65, Loss: 1.0954, Accuracy: 0.3125
Epoch 0, Step 70, Loss: 1.0928, Accuracy: 0.3750
Epoch 0, Step 75, Loss: 1.0808, Accuracy: 0.5625
Epoch 0, Step 80, Loss: 1.0748, Accuracy: 0.5000
Epoch 0, Step 85, Loss: 1.0268, Accuracy: 0.5625
Epoch 1, Step 0, Loss: 0.9706, Accuracy: 0.6875
Epoch 1, Step 5, Loss: 1.0557, Accuracy: 0.5000
Epoch 1, Step 10, Loss: 

KeyboardInterrupt: 